# SRD knowledge graph

Builds the graph the chatbot retrieves from and draws its nodes and edges.

- Nodes are the sections `markdown-chunkify` splits `srd_extraction.md` into.
- `CONTAINS` edges link a section to its direct subsections.
- `MENTIONS` edges link a section to a section whose header appears in its text.

The interactive views are written to `notebooks/output/` as HTML files. Open them in a browser if your notebook viewer doesn't render iframes.

In [1]:
import sys
from collections import Counter
from pathlib import Path

import networkx as nx
import pandas as pd
from IPython.display import HTML
from IPython.display import display
from pyvis.network import Network

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

from src.graphrag.graph import CONTAINS  # noqa: E402
from src.graphrag.graph import MENTIONS  # noqa: E402
from src.graphrag.graph import build_graph  # noqa: E402

OUTPUT_DIR = REPO_ROOT / "notebooks" / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

graph = build_graph(REPO_ROOT / "srd_extraction.md")
print(f"{graph.number_of_nodes():,} nodes, {graph.number_of_edges():,} edges")

2,954 nodes, 12,941 edges


## Nodes and relationships

Node counts per chapter, then edge counts by type and by the chapters they connect.

In [2]:
nodes = pd.DataFrame.from_dict(dict(graph.nodes(data=True)), orient="index")
edges = pd.DataFrame(
    [
        {
            "source": graph.nodes[source]["category"],
            "target": graph.nodes[target]["category"],
            "type": data["type"],
        }
        for source, target, data in graph.edges(data=True)
    ]
)

display(nodes["category"].value_counts().rename("sections").to_frame())
display(edges["type"].value_counts().rename("edges").to_frame())

,sections
category,
Monsters,1298
Spells,379
Classes,353
Magic Items,297
Equipment,187
Rules Glossary,158
Playing the Game,105
Gameplay Toolbox,81
Character Creation,43


,edges
type,
MENTIONS,9998
CONTAINS,2943


In [3]:
# Rows mention columns: how often sections of one chapter refer to sections of another.
mentions = edges[edges["type"] == MENTIONS]
pd.crosstab(mentions["source"], mentions["target"])

target,Character Creation,Character Origins,Classes,Equipment,Feats,Gameplay Toolbox,Magic Items,Monsters,Playing the Game,Rules Glossary,Spells
source,,,,,,,,,,,
Character Creation,8,13,70,2,5,0,0,2,4,7,8
Character Origins,4,5,3,2,11,3,0,0,0,51,22
Classes,52,0,139,31,70,4,4,11,15,373,1010
Equipment,0,0,6,26,2,6,9,4,0,146,9
Feats,0,17,2,4,0,0,0,0,0,17,1
Gameplay Toolbox,0,0,1,3,1,18,1,4,4,86,21
Magic Items,1,2,36,20,1,17,59,22,4,470,209
Monsters,1,0,5,8,0,118,1,924,19,2064,444
Playing the Game,2,1,4,4,0,1,1,2,25,164,6


In [4]:
# The most referenced sections.
in_degree = pd.Series(dict(graph.in_degree()), name="mentioned by")
nodes.join(in_degree).nlargest(15, "mentioned by")[["breadcrumb", "mentioned by"]]

,breadcrumb,mentioned by
752,Spells > Casting Spells > Casting Time,341
757,Spells > Casting Spells > Components,341
1229,Rules Glossary > Rules Definitions > Passive P...,340
1675,Monsters > Parts of a Stat Block > Languages,340
1134,Rules Glossary > Rules Definitions > Attack Roll,338
1125,Rules Glossary > Rules Definitions > Action,297
479,Classes > Wizard,257
1668,Monsters > Parts of a Stat Block > Skills,220
1220,Rules Glossary > Rules Definitions > Magic [Ac...,209
1126,Rules Glossary > Rules Definitions > Advantage,193


## Full graph

Every node and edge. Node color is the chapter, node size grows with the number of connections. Dark edges are `CONTAINS`, light edges are `MENTIONS`. Hover a node for its breadcrumb and an edge for its type. Scroll to zoom, drag to pan.

The layout is computed once with networkx and physics is turned off, so the browser stays responsive with all edges drawn. Edges are hidden while you drag.

In [5]:
# The seven largest chapters get a color each, the rest share gray.
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6"]
OTHER_COLOR = "#8a8a85"
EDGE_COLORS = {CONTAINS: "#3d3d3a", MENTIONS: "#b5b5b0"}
FAINT_MENTIONS = "rgba(120, 120, 115, 0.12)"  # 10,000 overlapping edges need transparency

largest = [category for category, _ in Counter(nodes["category"]).most_common(len(PALETTE))]
category_colors = dict(zip(largest, PALETTE))


def node_color(category: str) -> str:
    return category_colors.get(category, OTHER_COLOR)


def legend() -> HTML:
    swatch = (
        '<span style="display:inline-block;width:12px;height:12px;border-radius:50%;'
        'background:{}"></span> {}'
    )
    line = '<span style="display:inline-block;width:18px;border-top:3px solid {}"></span> {}'
    items = [swatch.format(color, name) for name, color in category_colors.items()]
    items.append(swatch.format(OTHER_COLOR, "Other chapters"))
    items += [line.format(color, edge_type) for edge_type, color in EDGE_COLORS.items()]
    return HTML(
        '<div style="display:flex;flex-wrap:wrap;gap:6px 18px">'
        + "".join(f'<span style="white-space:nowrap">{item}</span>' for item in items)
        + "</div>"
    )


def draw(subgraph: nx.DiGraph, filename: str, positions: dict | None = None) -> HTML:
    """Write a subgraph to an interactive HTML file and return an iframe showing it.

    With `positions` the nodes are pinned and physics is off. Without, vis.js lays the
    graph out and edges are labeled with their type.
    """
    net = Network(height="750px", width="100%", directed=True, cdn_resources="in_line")
    for node, data in subgraph.nodes(data=True):
        options = {}
        if positions is not None:
            x, y = positions[node]
            options = {"x": float(x) * 4000, "y": float(y) * 4000, "physics": False}
        net.add_node(
            node,
            label=data["header"] if positions is None else " ",
            title=data["breadcrumb"],
            color=node_color(data["category"]),
            size=8 + 2 * subgraph.degree(node) ** 0.5,
            **options,
        )
    for source, target, data in subgraph.edges(data=True):
        options = {"color": EDGE_COLORS[data["type"]]}
        if positions is None:
            options["label"] = data["type"]
            options["font"] = {"size": 9, "color": "#6b6b66", "strokeWidth": 3}
        elif data["type"] == MENTIONS:
            options["color"] = FAINT_MENTIONS
        net.add_edge(source, target, title=data["type"], width=1, **options)
    if positions is not None:
        net.set_options(
            '{"physics": {"enabled": false}, "edges": {"smooth": false},'
            ' "interaction": {"hideEdgesOnDrag": true}}'
        )

    path = OUTPUT_DIR / filename
    path.write_text(net.generate_html(notebook=False), encoding="utf-8")
    relative = path.relative_to(Path.cwd()) if path.is_relative_to(Path.cwd()) else path
    return HTML(
        f'<iframe src="{relative.as_posix()}" width="100%" height="770"'
        ' style="border:1px solid #ccc"></iframe>'
    )

In [6]:
# Pulling harder along CONTAINS edges keeps each chapter's sections together.
layout_graph = graph.to_undirected()
for _, _, data in layout_graph.edges(data=True):
    data["weight"] = 5 if data["type"] == CONTAINS else 0.2
positions = nx.spring_layout(layout_graph, seed=7, iterations=100, weight="weight")

display(legend())
draw(graph, "srd_graph.html", positions)

/tmp/claude-0/-home-user/09548ae6-6723-54db-a98e-1d7590b802d6/scratchpad/venv/lib/python3.12/site-packages/IPython/core/display.py:448: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


## Neighborhood of one section

The full graph shows the overall shape. To read individual relationships, draw one section with everything it links to and everything that links to it. Edges are labeled with their type.

In [7]:
def draw_neighborhood(header: str, radius: int = 1) -> HTML:
    """Draw the sections within `radius` hops of the first section with this header."""
    center = next(node for node, data in graph.nodes(data=True) if data["header"] == header)
    neighborhood = nx.ego_graph(graph, center, radius=radius, undirected=True)
    print(f"{neighborhood.number_of_nodes()} nodes, {neighborhood.number_of_edges()} edges")
    return draw(neighborhood, f"neighborhood_{header.lower().replace(' ', '_')}.html")


display(legend())
draw_neighborhood("Fireball")

29 nodes, 38 edges


In [8]:
display(legend())
draw_neighborhood("Paralyzed [Condition]")

67 nodes, 84 edges
